# 02_finetune_stage1 — Stage 1 Fine-Tune (MIMIC → ICD + Medications + Query)

Trains BioMistral-7B (4-bit NF4) with QLoRA to predict, from a patient record:
- `icd_titles` — ICD diagnosis titles
- `adm_medications` — medications administered during the visit
- `retrieval_query` — a short query for downstream CREST retrieval

**Dataset**: MIMIC-IV Demo (126 rows, 90/10 split by `stay_id`).
**Adapter**: saved to `DRIVE_PATH/stage1_adapter/`.


In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

!pip install -q --no-cache-dir \
    torchvision==0.19.0 \
    torchaudio==2.4.0 \
    transformers==4.51.0 \
    peft==0.12.0 \
    trl==0.10.1 \
    bitsandbytes==0.43.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 96.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 229.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 195.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 399.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.1/280.1 kB 395.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 MB 230.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 MB 118.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 122.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 111.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 124.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 238.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 104.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os
import pickle
from kaggle_secrets import UserSecretsClient
import huggingface_hub

# Paths
WORKING_DIR = "/kaggle/working"
os.makedirs(WORKING_DIR, exist_ok=True)

# HuggingFace login
secrets = UserSecretsClient()
huggingface_hub.login(
    token=secrets.get_secret("HF_TOKEN"),
    add_to_git_credential=False
)

In [3]:
import sys
sys.path.append("/kaggle/input/datasets/aanyagupta0921/senior-project-utils")

from utils import (
    load_biomistral_4bit,
    format_stage1_prompt,
    format_stage2_prompt,
    build_rag_index,
    retrieve,
    render_patient_note,
)

In [4]:
import ast
import json
import pandas as pd

mimic_df = pd.read_csv("/kaggle/input/datasets/aanyagupta0921/senior-project-mimic-dataset/mimic_data.csv")

LIST_COLS = ["icd_title", "med_record", "adm_med_name", "chiefcomplaint", "etcdescription"]
for col in LIST_COLS:
    mimic_df[col] = mimic_df[col].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

print(f"Loaded {len(mimic_df)} MIMIC rows")
mimic_df[["stay_id", "icd_title", "adm_med_name", "chiefcomplaint"]].head(3)


Loaded 126 MIMIC rows


,stay_id,icd_title,adm_med_name,chiefcomplaint
0,30094124,"[Weakness, Dehydration]","[Lidocaine Viscous 2% 15mL UDCUP, Aluminum-Mag...","[Fatigue, s/p Fall]"
1,30225689,[Unspecified abdominal pain],"[Morphine Sulfat 4mg/1mL 1mL VIAL, Readi-Cat 2...",[Right sided abdominal pain]
2,30279522,[Weakness],"[Acyclovir, Hydrocodone-Acetamin(5mg-325mg), M...",[Weakness]


In [5]:
from sklearn.model_selection import train_test_split

stay_ids = mimic_df["stay_id"].unique()
train_ids, val_ids = train_test_split(stay_ids, test_size=0.1, random_state=42)
train_df = mimic_df[mimic_df["stay_id"].isin(train_ids)].reset_index(drop=True)
val_df   = mimic_df[mimic_df["stay_id"].isin(val_ids)].reset_index(drop=True)
print(f"Train: {len(train_df)} rows  |  Val: {len(val_df)} rows")


def make_stage1_example(row):
    patient_note    = render_patient_note(row)
    icd_titles      = row["icd_title"]     # already a list
    adm_meds        = row["adm_med_name"]  # already a list
    retrieval_query = f"guidelines for {', '.join(t.lower() for t in icd_titles)}"
    target = json.dumps({
        "icd_titles":      icd_titles,
        "retrieval_query": retrieval_query,
        "adm_medications": adm_meds,
    })
    return format_stage1_prompt(patient_note, response=target)


train_texts = [make_stage1_example(row) for _, row in train_df.iterrows()]
val_texts   = [make_stage1_example(row) for _, row in val_df.iterrows()]
print(f"Built {len(train_texts)} train + {len(val_texts)} val examples")
print("\nSample:")
print(train_texts[0])


Train: 113 rows  |  Val: 13 rows
Built 113 train + 13 val examples

Sample:
<s>[INST] You are a clinical decision support assistant. Given a patient record, output a JSON object with exactly three keys:
  "icd_titles": list of ICD diagnosis titles
  "retrieval_query": short query string for retrieving relevant clinical guidelines
  "adm_medications": list of medications administered during the visit

Patient Record
- Gender: MALE
- Race: WHITE
- Disposition: ADMITTED
- Chief complaint: Fatigue, s/p Fall
- Acuity: 2
- Pain score: 4
- Medication history: metoprolol tartrate, albuterol sulfate, aspirin, fenofibrate, ziprasidone HCl, levothyroxine, acyclovir, lidocaine, trazodone, gabapentin, losartan, furosemide, insulin glargine [Lantus], pantoprazole, fluoxetine, acetaminophen [Mapap (acetaminophen)], clopidogrel, hydrocodone-acetaminophen, divalproex [Depakote], docusate sodium, ondansetron HCl, metformin, magnesium oxide, melatonin, sennosides [senna], atorvastatin, ibuprofen

Vitals 

In [6]:
from datasets import Dataset

train_dataset = Dataset.from_dict({"text": train_texts})
val_dataset   = Dataset.from_dict({"text": val_texts})
print(train_dataset)


Dataset({
    features: ['text'],
    num_rows: 113
})


In [7]:

model, tokenizer = load_biomistral_4bit()
print("Model loaded. Trainable params before LoRA:")
print(sum(p.numel() for p in model.parameters() if p.requires_grad))


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

2026-05-08 20:29:54.966812: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778272195.369077      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778272195.465299      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778272196.379507      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778272196.379538      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778272196.379541      23 computation_placer.cc:177] computation placer alr

pytorch_model.bin:   0%|          | 0.00/14.5G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model loaded. Trainable params before LoRA:
262410240


In [8]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=32,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 27,262,976 || all params: 7,268,995,072 || trainable%: 0.3751


In [9]:
from transformers import TrainingArguments
from trl import SFTTrainer

STAGE1_ADAPTER_PATH = "/kaggle/working/stage1_adapter"

training_args = TrainingArguments(
    output_dir=STAGE1_ADAPTER_PATH,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    dataloader_pin_memory=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    max_seq_length=2048,
    dataset_text_field="text",
)

trainer.train()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': max_seq_length, dataset_text_field. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:283: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:321: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/113 [00:00<?, ? examples/s]

Map:   0%|          | 0/13 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:412: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:600: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/

Epoch,Training Loss,Validation Loss
1,0.676400,0.620826
2,0.479600,0.524062


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:600: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:600: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an except

TrainOutput(global_step=42, training_loss=0.7259428628853389, metrics={'train_runtime': 954.1837, 'train_samples_per_second': 0.355, 'train_steps_per_second': 0.044, 'total_flos': 7723261315596288.0, 'train_loss': 0.7259428628853389, 'epoch': 2.849557522123894})

In [10]:
trainer.model.save_pretrained(STAGE1_ADAPTER_PATH)
tokenizer.save_pretrained(STAGE1_ADAPTER_PATH)
print(f"Stage 1 adapter saved to {STAGE1_ADAPTER_PATH}")


Stage 1 adapter saved to /kaggle/working/stage1_adapter


In [11]:
import torch

model.eval()

sample_row    = val_df.iloc[0]
patient_note  = render_patient_note(sample_row)
prompt        = format_stage1_prompt(patient_note)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

generated = tokenizer.decode(
    output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
).strip()

print("=== Raw output ===")
print(generated)
print()

# Strip any trailing </s> that wasn't caught by skip_special_tokens
generated_clean = generated.split("</s>")[0].strip()

try:
    parsed = json.loads(generated_clean)
    print("=== Parsed JSON ===")
    print(f"  icd_titles:      {parsed.get('icd_titles', [])}")
    print(f"  retrieval_query: {parsed.get('retrieval_query', '')}")
    print(f"  adm_medications: {parsed.get('adm_medications', [])}")

    assert isinstance(parsed.get("icd_titles"), list) and parsed["icd_titles"], \
        "icd_titles missing or empty"
    assert isinstance(parsed.get("adm_medications"), list) and parsed["adm_medications"], \
        "adm_medications missing or empty"
    print("\nSanity check PASSED")
except (json.JSONDecodeError, AssertionError) as e:
    print(f"\nSanity check FAILED: {e}")
    print("Review raw output above — the model may need more epochs or the output may need regex extraction.")

print()
print("=== Ground truth ===")
print(f"  icd_titles:      {sample_row['icd_title']}")
print(f"  adm_medications: {sample_row['adm_med_name']}")


=== Raw output ===
{"icd_titles": ["T-SPINE FX DISPLACEMENT"], "retrieval_query": "guidelines for t-spine fx displacement", "adm_medications": ["Morphine Sulfate (MS Contin)", "Docusate Sodium", "Lorazepam", "Morphine (Roxicodone)", "Heparin (porcine)", "Fluticasone", "Tizanidine", "Loratadine", "Famotidine", "Melatonin"]}

=== Parsed JSON ===
  icd_titles:      ['T-SPINE FX DISPLACEMENT']
  retrieval_query: guidelines for t-spine fx displacement
  adm_medications: ['Morphine Sulfate (MS Contin)', 'Docusate Sodium', 'Lorazepam', 'Morphine (Roxicodone)', 'Heparin (porcine)', 'Fluticasone', 'Tizanidine', 'Loratadine', 'Famotidine', 'Melatonin']

Sanity check PASSED

=== Ground truth ===
  icd_titles:      ['Unsp fracture of T7-T8 vertebra, init for clos fx', 'Car occupant (driver) (passenger) injured in unsp traf, init', 'Unsp fracture of T5-T6 vertebra, init for clos fx']
  adm_medications: ['LORazepam 1mg TAB']
